<a href="https://colab.research.google.com/github/weiyuli20/rag_eval/blob/main/semantic_chunk.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 学习语义切分策略
reference:
https://zhuanlan.zhihu.com/p/1924496550433919103




# 相似度突变点分割

将段落切分为句子，为每个句子生成embedding, 计算相邻句子间的相似度，大于阈值就断开



In [7]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import re

# 1. 文本→句子（支持中英文简单分句）

def split_sentences(text):
    return [s.strip() for s in re.split(r'(?<=[。！？.!?])', text) if s.strip()]

# 2. 获得embedding
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')  # 支持多语言
# 例：也可用'moka-ai/m3e-base'等专门中文模型

def semantic_chunk(text, sim_threshold=0.70, min_chunk=2, max_chunk=8):
    """
    基于句子embedding的语义分块
    sim_threshold: 相邻句余弦相似度低于该阈值则切块
    min_chunk/max_chunk: 控制每块的最小/最大句数
    """
    sentences = split_sentences(text)
    embeddings = model.encode(sentences)

    chunks, chunk = [], []
    for i, sent in enumerate(sentences):
        chunk.append(sent)
        # 不是最后一句
        if i < len(sentences) - 1:
            sim = cosine_similarity([embeddings[i]], [embeddings[i+1]])[0][0]
            print(f"第{i}句 和第{i+1} 句的相似度为：{sim}")
            # 满足分割条件
            if sim < sim_threshold and len(chunk) >= min_chunk or len(chunk) >= max_chunk:
                chunks.append(''.join(chunk))
                chunk = []
    if chunk:
        chunks.append(''.join(chunk))
    return chunks

# ========== 示例 ==========
if __name__ == "__main__":
    text = """
AI大模型时代的到来，让很多行业发生了巨大变革。以Robo-Advisor为例，它让普通用户也能获得专业理财建议。这一过程中，AI技术的不断进步也让用户的使用体验不断提升。
不过，有些用户也担心自己的数据隐私问题。还有人觉得，AI给出的建议虽然方便，但有时不如真人顾问细致。
总体来看，未来AI在金融行业的渗透只会越来越深，谁能把握好用户需求，谁就能获得更多的市场份额。
与此同时，合规和安全问题也在引发监管部门的关注。各家金融科技公司都在积极探索如何更好地兼容创新与合规。
"""
    result = semantic_chunk(text, sim_threshold=0.3) # 可以调整阈值观察输出
    for i, chunk in enumerate(result):
        print(f"Chunk {i+1}:\n{chunk}\n{'-'*20}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

第0句 和第1 句的相似度为：0.10113576054573059
第1句 和第2 句的相似度为：0.2885872721672058
第2句 和第3 句的相似度为：0.17521341145038605
第3句 和第4 句的相似度为：0.2328348010778427
第4句 和第5 句的相似度为：0.4405905604362488
第5句 和第6 句的相似度为：0.3404782712459564
第6句 和第7 句的相似度为：0.32132184505462646
Chunk 1:
AI大模型时代的到来，让很多行业发生了巨大变革。以Robo-Advisor为例，它让普通用户也能获得专业理财建议。
--------------------
Chunk 2:
这一过程中，AI技术的不断进步也让用户的使用体验不断提升。不过，有些用户也担心自己的数据隐私问题。
--------------------
Chunk 3:
还有人觉得，AI给出的建议虽然方便，但有时不如真人顾问细致。总体来看，未来AI在金融行业的渗透只会越来越深，谁能把握好用户需求，谁就能获得更多的市场份额。与此同时，合规和安全问题也在引发监管部门的关注。各家金融科技公司都在积极探索如何更好地兼容创新与合规。
--------------------
